In [1]:
import os
import qt
from qt import dt, np, pd

import sys
sys.path.append('..')
from spx_history import *

In [2]:
ADJ_PX_CSV_DIR = "/mnt/f/data_adjusted_by_symbol/"

- universe and holidays

In [3]:
start_date = dt.date(2022,1,1)
end_date = dt.date(2024,12,31)

In [4]:
univ = pd.read_csv('../spy_weight_historical.csv')
univ['date'] = pd.to_datetime(univ['date']).dt.date
univ = univ[(univ['date'] >= start_date) & (univ['date'] <= end_date)].copy()
univ = univ.sort_values('date').reset_index(drop=True)

In [5]:
tickers_univ = univ['ticker'].unique().tolist()
len(tickers_univ)

570

- read adjusted history between period

In [34]:
sym = 'AAPL'

In [35]:
t_sym = pd.read_csv(f'{ADJ_PX_CSV_DIR}/{sym}.csv')

In [36]:
t = t_sym.copy()

# drop duplicate by times
t = t.groupby('datetime_us').last().reset_index()

# add datetime_us, date, time
t['date'] = pd.to_datetime(t['date']).dt.date
t['datetime_us'] = pd.to_datetime(t['datetime_us'])
t['time'] = pd.to_datetime(t['datetime_us']).dt.time

# filter for relevant dates
t = t[(t['date'] >= start_date) & (t['date'] <= end_date)]
t = t[['datetime_us', 'date', 'time', 'open', 'high', 'low', 'close', 'volume', 'adj_cum']]
t = t[pd.to_datetime(t['date']).dt.dayofweek < 5].sort_values('datetime_us').reset_index(drop=True)

In [37]:
ret_horizons = [1,2]
# ret_horizons = [1,2,3,4,5,10,15,30,60]

for h in ret_horizons:
	t[f'ret_{h}m'] = 1e4*np.log(t['close'] / t.groupby('date')['close'].shift(h)).fillna(0)
	t[f'fwd_ret_{h}m'] = 1e4*np.log(t.groupby('date')['close'].shift(-h) / t['close']).fillna(0)

t['z_score_ret'] = t.groupby('date')['ret_1m'].transform(lambda x: (x - x.mean()) / x.std())
t['z_score_vol'] = t.groupby('date')['volume'].transform(lambda x: (x - x.mean()) / x.std())

In [38]:
threshold = 200
small_epsilon = 25
t['is_bad_tick'] = (t['ret_2m'].abs() > threshold) & ((t['ret_2m'] + t['fwd_ret_2m']).abs() < small_epsilon)

In [39]:
t.loc[t['is_bad_tick'] == True, 'close'] = np.nan
t['close'] = t.groupby('date')['close'].ffill()

In [48]:
# t[t['date'] <= dt.date(2023, 1, 1)] .set_index('datetime_us')['close'].plot()

In [49]:
t[t['date'].isin(t[t['is_bad_tick']]['date'].unique())].sort_values('datetime_us').set_index('datetime_us') ['close'].plot()

- analyze a date for outliers

In [42]:
query_date = dt.date(2024,4,25)

In [43]:
qt.view(t[t['date'] == query_date])

Grid(columns_fit='auto', compress_data=True, css_rules_down=['.number-cell {text-align: left;width: 10;}', '.l…

In [44]:
# t.set_index('datetime_us').sort_index()['close'].plot()

In [45]:
t[t['date'] == query_date].set_index('datetime_us').sort_index() ['close'] .plot()
# t[t['date'] == query_date].set_index('datetime_us').sort_index() ['volume'] .plot()
t[t['date'] == query_date].set_index('datetime_us').sort_index() [['ret_1m', 'z_score_ret', 'z_score_vol']] .plot()

In [ ]:
t[t['z_score_ret'].abs() > 3] .plot.scatter(x='z_score_ret', y='z_score_vol')

In [ ]:
t

In [ ]:
t['z_score_ret'].plot.hist()

In [ ]:
tsum = t[
	(t['time'] >= dt.time(9,30)) &
	(t['time'] <= dt.time(16,0))
] .groupby('date').agg(
	count_rows = ('ret_1m', 'count'),
	count_zero_ret_1m = ('ret_1m', lambda x: (x==0).sum()),
	mean_ret_1m = ('ret_1m', 'mean'),
	std_ret_1m = ('ret_1m', 'std'),
	max_ret_1m = ('ret_1m', 'max'),
	min_ret_1m = ('ret_1m', 'min'),
	max_min_ret_1m = ('ret_1m', lambda x: x.max() - x.min())
)
# qt.view(tsum)
tsum['std_ret_1m'].plot()
tsum['max_min_ret_1m'].plot()

In [ ]:
tsum.loc[dt.date(2022,4,19)]

In [ ]:
tbt = t[t['is_bad_tick']].sort_values('datetime_us')
tbt[
	(tbt['time'] >= dt.time(9,30)) &
	(tbt['time'] <= dt.time(16,0))
]

In [ ]:
t[
	(t['z_score_ret'].abs() > 3) &
	(t['z_score_vol'].abs() < 1)
]